# **GAP NOTES**

(topics identified as missing from existing notebooks - added the night before the exam)

<div class="alert alert-block alert-info">

### **Gap 1 - NumPy Broadcasting Formal Rules**

Topic: NumPy / Week 2–3 level
</div>

<div class="alert alert-block alert-danger">

##### **G1.1 The 4 broadcasting rules**

</div>

NumPy broadcasting allows operations on arrays with different shapes. The rules are applied **right-to-left** (align from the trailing dimension):

> **Rule 1** - If the arrays have different numbers of dimensions, the shape of the smaller-dimensional array is **padded with 1s on the left**.

> **Rule 2** - Arrays with a size of **1 along a particular dimension** act as if they had the size of the array with the largest size in that dimension.

> **Rule 3** - If neither the sizes match nor either size is 1, broadcasting **raises an error**.

> **Rule 4** - Arrays must be compatible in every dimension after padding (either equal sizes, or one of them is 1).

One-liner to remember: **"align right, a 1 can stretch, anything else must match exactly, mismatch = error."**

<div class="alert alert-block alert-danger">

##### **G1.2 Broadcasting worked examples**

</div>

**Example 1 - missing dimension padded with 1:**
```
A: shape (3, 4)
B: shape    (4,)   →  padded to (1, 4)
             ↓
result: (3, 4)   ✓  B stretched from (1,4) to (3,4)
```

**Example 2 - two 1s stretch in different dimensions:**
```
A: shape (3, 1)
B: shape (1, 4)
             ↓
result: (3, 4)   ✓  A stretches cols, B stretches rows
```

**Example 3 - three dimensions, one padded:**
```
A: shape (3, 1, 4)
B: shape    (5, 4)  →  padded to (1, 5, 4)
             ↓
result: (3, 5, 4)   ✓
```

**Example 4 - incompatible, raises error:**
```
A: shape (3,)
B: shape (4,)
             ↓
ERROR: shapes (3,) and (4,) not aligned   ✗
```

**Example 5 - scalar always broadcasts:**
```
A: shape (100, 200)
B: scalar = 3.14    →  treated as shape ()
             ↓
result: (100, 200)   ✓  scalar stretches to any shape
```

<div class="alert alert-block alert-danger">

##### **G1.3 Quick compatibility truth table**

</div>

| Shape A | Shape B | Result shape | OK? |
|---|---|---|---|
| `(N,)` | `(N,)` | `(N,)` | ✓ |
| `(N,)` | `(1,)` | `(N,)` | ✓ |
| `(N,)` | `(M,)` N≠M | - | ✗ error |
| `(M, N)` | `(N,)` | `(M, N)` | ✓ padded to `(1, N)` |
| `(M, N)` | `(M, 1)` | `(M, N)` | ✓ col broadcast |
| `(M, N)` | `(1, N)` | `(M, N)` | ✓ row broadcast |
| `(M, N)` | `(M, K)` N≠K | - | ✗ error |
| `(3, 1, 4)` | `(5, 4)` | `(3, 5, 4)` | ✓ |

If you have a computer in the exam: `np.broadcast_shapes((3,1,4), (5,4))` returns the result shape immediately, or just create small arrays and try the operation.

<div class="alert alert-block alert-info">

### **Gap 2 - `time` command: `real` / `user` / `sys` interpretation**

Topic: Profiling / Parallelism diagnosis
</div>

<div class="alert alert-block alert-danger">

##### **G2.1 What each field means**

</div>

Running `time python script.py` gives:

```bash
real    0m4.231s
user    0m15.847s
sys     0m0.212s
```

> **`real`** - wall-clock time. How long you actually waited. Includes I/O waits, sleeping, and time when the process was not scheduled.

> **`user`** - total CPU time spent in user-space (your Python code, NumPy, etc.). **Summed across all cores.** A 4-core job where all cores are busy for 4 seconds shows `user ≈ 16s`.

> **`sys`** - total CPU time spent in kernel/OS calls (memory allocation, I/O system calls, etc.). Usually small. Large `sys` means lots of OS overhead.

<div class="alert alert-block alert-danger">

##### **G2.2 Diagnosis table - what does the ratio tell you?**

</div>

| Observation | Meaning | Likely cause |
|---|---|---|
| `user ≈ p × real` | Good parallelism, all `p` cores busy | Multiprocessing/BLAS working correctly |
| `user ≈ real` | Effectively single-threaded | GIL blocking threads, or only 1 core used |
| `user < real` | Less CPU than wall time | I/O bound - waiting for disk/network |
| `user >> real` | More CPU than wall time | Multi-core (expected with parallelism) |
| `sys` large | Heavy OS interaction | Lots of small allocations, excessive I/O calls |

**Concrete example - 4-core parallel job:**
```
real    0m5.2s    ← you waited 5 seconds
user    0m19.8s   ← 4 cores × ~5s each ≈ 20s of CPU work
sys     0m0.1s
```
This shows good parallelism: `user / real ≈ 3.8` (close to 4 cores).

**Example - GIL blocking threads:**
```
real    0m8.4s
user    0m8.3s   ← only 1 core worth of work despite 8 threads
sys     0m0.2s
```
This shows the GIL is preventing real parallelism: `user ≈ real`.

> KEY RULE: `user / real` estimates how many cores were effectively used. If this ratio is much less than the number of cores you requested, something is wrong - GIL, undersubscription, or I/O waiting.

<div class="alert alert-block alert-info">

### **Gap 3 - `pd.read_csv(chunksize=N)` returns an iterator, not a DataFrame**

Topic: I/O / pandas / Week 7
</div>

<div class="alert alert-block alert-danger">

##### **G3.1 What `chunksize` actually returns**

</div>

When you pass `chunksize=N` to `pd.read_csv`, it does **not** return a DataFrame. It returns a `TextFileReader` iterator object. You must iterate over it:

```python
# CORRECT usage - iterate over chunks:
for chunk in pd.read_csv('big_file.csv', chunksize=100_000):
    # chunk is a normal DataFrame with up to 100,000 rows
    process(chunk)

# ALSO CORRECT - explicitly convert to list of DataFrames:
chunks = list(pd.read_csv('big_file.csv', chunksize=100_000))
df = pd.concat(chunks)   # but this loads everything into memory at once - defeats the purpose
```

```python
# WRONG - this will fail:
df = pd.read_csv('big_file.csv', chunksize=100_000)
df.head()     # AttributeError: 'TextFileReader' object has no attribute 'head'
df.shape      # AttributeError
df['col']     # AttributeError
```

> KEY RULE: `pd.read_csv(chunksize=N)` → `TextFileReader` (iterator). Each iteration yields a DataFrame of at most N rows. Never assign it to `df` and use it as if it were a DataFrame.

<div class="alert alert-block alert-danger">

##### **G3.2 When to use chunksize**

</div>

Use `chunksize` when the file is **larger than available RAM**. The pattern is:

```python
# Aggregate something without loading the full file:
total = 0
for chunk in pd.read_csv('huge.csv', chunksize=500_000):
    total += chunk['value'].sum()
```

This keeps at most 500,000 rows in memory at any moment. The garbage collector frees each chunk after the loop body finishes.

**Alternatives when chunksize is not the right tool:**

| Situation | Better approach |
|---|---|
| File fits in RAM, just slow to load | Use `pyarrow.csv.read_csv` (3.7× faster) |
| File loaded repeatedly | Convert to Parquet once, read Parquet |
| Array data, not tabular | Use `np.memmap` |
| Chunked array storage | Use `zarr` |

<div class="alert alert-block alert-info">

### **Gap 4 - Numba `@vectorize` target options**

Topic: Numba / Week 9–12
</div>

<div class="alert alert-block alert-danger">

##### **G4.1 The three `target` options**

</div>

`@vectorize` turns a scalar function into a NumPy ufunc that applies element-wise to arrays. The `target` argument controls where and how it runs:

| `target` | Execution | When to use |
|---|---|---|
| `"cpu"` | Single-threaded CPU (default) | Small arrays, or when overhead of parallelism isn't worth it |
| `"parallel"` | Multi-threaded CPU (uses all cores) | Large arrays, element-wise work, CPU-bound |
| `"cuda"` | GPU (one CUDA thread per element) | Very large arrays, GPU available, arithmetic-heavy |

```python
from numba import vectorize, float64

# Single-threaded - safe default
@vectorize([float64(float64, float64)], target='cpu')
def add_cpu(a, b):
    return a + b

# Multi-threaded CPU - good for large arrays
@vectorize([float64(float64, float64)], target='parallel')
def add_parallel(a, b):
    return a + b

# GPU - each element handled by one CUDA thread
@vectorize([float64(float64, float64)], target='cuda')
def add_cuda(a, b):
    return a + b
```

All three are called the same way - like a normal NumPy operation:
```python
result = add_parallel(x, y)   # x, y are NumPy arrays
```

> KEY ADVANTAGE over `@cuda.jit`: no manual index management, no bounds checks, no grid/block launch syntax. `@vectorize(target='cuda')` handles all of that automatically. Use it when the operation is truly element-wise (no dependencies between elements).

<div class="alert alert-block alert-danger">

##### **G4.2 `@vectorize` vs `@cuda.jit` - when to use which**

</div>

| Tool | Use when | Notes |
|---|---|---|
| `@vectorize(target='cuda')` | Pure element-wise scalar function on GPU | No index management, NumPy-like call syntax |
| `@cuda.jit` | Custom kernel: reductions, 2D, shared memory, non-trivial access patterns | Manual `cuda.grid()`, bounds check, launch config |
| `@njit(parallel=True)` + `prange` | CPU-only parallel loops | No GPU, but very simple to apply |

**Exam question type:** *"You have a scalar function applied element-wise to a 10M-element array. Which is the most appropriate Numba tool for running it on the GPU?"*

→ **`@vectorize(target='cuda')`** - it is exactly designed for this case. `@cuda.jit` would work too but requires boilerplate.

<div class="alert alert-block alert-info">

### **Gap 5 - Floor vs ceiling division in CUDA kernel launch**

Topic: GPU / Week 9–10
</div>

<div class="alert alert-block alert-danger">

##### **G5.1 The correct formula and what goes wrong without it**

</div>

When launching a 1D CUDA kernel for N elements with `tpb` threads per block:

```python
tpb = 512
n   = 1000

# CORRECT - ceiling division ensures all elements are covered:
bpg = (n + tpb - 1) // tpb   # = (1000 + 511) // 512 = 1511 // 512 = 2 blocks
# 2 blocks × 512 threads = 1024 threads, covers all 1000 elements (24 threads idle)

# WRONG - floor division silently skips the last batch:
bpg = n // tpb                # = 1000 // 512 = 1 block
# 1 block × 512 threads = only 512 threads → elements 512–999 are NEVER processed
# No error is raised. The output array has garbage/zeros in those positions.
```

> CONSEQUENCE of floor division: **silent wrong results**. The kernel runs without error, but the last `n % tpb` elements (here: 488 elements) are silently skipped. This is one of the hardest bugs to catch because there is no exception.

This is why the bounds check inside the kernel is also required:

```python
@cuda.jit
def kernel(x, y):
    i = cuda.grid(1)
    if i < len(x):        # ← bounds check: the extra 24 threads do nothing
        y[i] = x[i] * 2
```

Without the bounds check, those extra threads would write beyond the array end - undefined behaviour / memory corruption.

<div class="alert alert-block alert-danger">

##### **G5.2 The two errors together - summary**

</div>

| Mistake | What happens | Detectable? |
|---|---|---|
| Floor division for `bpg` | Last `n % tpb` elements skipped | ✗ Silent - wrong output, no error |
| No bounds check in kernel | Threads beyond `n` write to invalid memory | ✗ Silent corruption or crash |
| Both correct | All elements processed, no out-of-bounds writes | ✓ Correct |

**Always pair ceiling division for launch config with a bounds check inside the kernel.** They solve complementary problems:
- Ceiling division: ensures every element has at least one thread assigned.
- Bounds check: ensures the extra padding threads (beyond n) do not write garbage.

<div class="alert alert-block alert-info">

### **Gap 6 - LSF output filenames: `%J` vs `%I`**

Topic: HPC / Job arrays / Week 1 + 11
</div>

<div class="alert alert-block alert-danger">

##### **G6.1 What `%J` and `%I` expand to**

</div>

In LSF job script `-o` and `-e` directives, two tokens are substituted at runtime:

| Token | Expands to | Same for all array elements? |
|---|---|---|
| `%J` | The **parent job ID** (a single integer, e.g. `4521372`) | ✓ Yes - same across the whole array |
| `%I` | The **array element index** (the value of `$LSB_JOBINDEX`) | ✗ No - unique per element (1, 2, 3, ...) |

```bash
# Regular job (no array) - use %J for a unique file per submission:
#BSUB -o output_%J.out    # e.g. output_4521372.out
#BSUB -e errors_%J.err

# Job array - use %I to get one file per array element:
#BSUB -J simulate[1-100]
#BSUB -o output_%I.out    # e.g. output_1.out, output_2.out, ..., output_100.out
#BSUB -e errors_%I.err

# Can also combine both - unique per submission AND per element:
#BSUB -o output_%J_%I.out  # e.g. output_4521372_37.out
```

> KEY TRAP: Using `%J` in a job array gives **all 100 elements the same output filename** - they all write to `output_4521372.out` and clobber each other. The fix is `%I`.

<div class="alert alert-block alert-danger">

##### **G6.2 The full picture - tokens available in LSF directives**

</div>

```bash
#BSUB -J myjob[1-50]              # defines the array, indices 1 to 50
#BSUB -o logs/out_%J_%I.out       # %J = parent job ID, %I = array index
#BSUB -e logs/err_%J_%I.err

# Inside the script, the index is available as:
echo $LSB_JOBINDEX                 # e.g. 37 (for the 37th element)

# In Python:
import sys
job_index  = int(sys.argv[1])      # passed from bash: python script.py $LSB_JOBINDEX
array_index = job_index - 1        # convert 1-based LSF → 0-based Python
```

**Alternative - pass the index directly from bash without sys.argv:**
```bash
python -u script.py $LSB_JOBINDEX 1> logs/out_${LSB_JOBINDEX}.out 2> logs/err_${LSB_JOBINDEX}.err
```
Here `%I` is replaced by LSF, but `$LSB_JOBINDEX` is a bash variable - use `${}` syntax in the redirect, not `%I`.